# Project Lab: Five Statistics Portfolio Projects

Notebook for **Chapter 15** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to run a project in this area

## 15.2 Project: Bootstrap and Confidence-Interval Explorer

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(15)
times = np.round(rng.lognormal(mean=6.0, sigma=0.5, size=300), 1)
print("n =", len(times), " mean =", round(times.mean(), 1), " median =", round(np.median(times), 1),
      " p90 =", round(np.percentile(times, 90), 1))

def bootstrap_ci(x, stat, B=5000, level=0.95, seed=0):
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(x), size=(B, len(x)))      # B resamples, with replacement
    reps = stat(x[idx], axis=1)                         # statistic of each resample
    lo, hi = np.percentile(reps, [(1-level)/2*100, (1+level)/2*100])
    return reps, lo, hi

p90 = lambda a, axis: np.percentile(a, 90, axis=axis)
for name, f in [("mean", np.mean), ("median", np.median), ("p90", p90)]:
    reps, lo, hi = bootstrap_ci(times, f)
    print(name, "boot SE =", round(reps.std(ddof=1), 1), " 95% CI =", (round(lo, 1), round(hi, 1)))
m, se = times.mean(), times.std(ddof=1)/np.sqrt(len(times))
tcrit = stats.t.ppf(0.975, len(times)-1)
print("t-interval mean:", round(m - tcrit*se, 1), round(m + tcrit*se, 1))

In [ ]:
true = {"median": np.exp(6.0), "p90": np.exp(6.0 + 0.5*1.2815516)}
fs = {"median": np.median, "p90": p90}
def coverage(stat, n, trials=400, B=1000):
    hit = 0
    for t in range(trials):
        x = rng.lognormal(6.0, 0.5, n)
        _, lo, hi = bootstrap_ci(x, fs[stat], B=B, seed=t)
        hit += (lo <= true[stat] <= hi)
    return hit/trials
for n in (10, 30, 100, 300):
    print("n =", n, " coverage median:", coverage("median", n), " coverage p90:", coverage("p90", n))

n = 10  coverage median: 0.93  coverage p90: 0.6875
n = 30  coverage median: 0.95  coverage p90: 0.8875
n = 100  coverage median: 0.9425  coverage p90: 0.9275
n = 300  coverage median: 0.9425  coverage p90: 0.9575


## 15.3 Project: Regression Diagnostics Report

In [ ]:
import numpy as np
from scipy import stats
def ols(X, y):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    fit = X @ beta; res = y - fit
    n, k = X.shape
    s2 = res @ res / (n-k)
    XtXi = np.linalg.inv(X.T @ X)
    h = np.einsum('ij,jk,ik->i', X, XtXi, X)          # leverage
    se = np.sqrt(s2*np.diag(XtXi))
    r2 = 1 - res@res/((y-y.mean())@(y-y.mean()))
    cook = (res**2/(k*s2))*h/(1-h)**2
    stud = res/np.sqrt(s2*(1-h))
    return dict(beta=beta, se=se, res=res, fit=fit, h=h, cook=cook, stud=stud, r2=r2, s2=s2, k=k)

def vif(X):                                           # regress each column on the others
    return [float(1/(1-ols(np.delete(X, j, axis=1), X[:, j])['r2'])) for j in range(1, X.shape[1])]

def bp_test(X, res):                                  # Breusch-Pagan: do squared residuals depend on X?
    r = ols(X, res**2); lm = len(res)*r['r2']
    return lm, stats.chi2.sf(lm, X.shape[1]-1)

## 15.4 Project: Bayesian A/B Test Calculator

In [ ]:
import numpy as np
from scipy import stats
def ab(sa, na, sb, nb, a0=1, b0=1, draws=200_000, seed=1):
    r = np.random.default_rng(seed)
    pa = stats.beta(a0+sa, b0+na-sa); pb = stats.beta(a0+sb, b0+nb-sb)
    da, db = pa.rvs(draws, random_state=r), pb.rvs(draws, random_state=r)
    return dict(ciA=pa.interval(0.95), ciB=pb.interval(0.95),
        p_b_better=(db > da).mean(),
        lift_ci=tuple(np.percentile((db-da)/da, [2.5, 97.5])),
        loss_choose_B=np.maximum(da-db, 0).mean(),
        loss_choose_A=np.maximum(db-da, 0).mean())

def decide(r, thr=0.95, eps=0.0005):
    if r['p_b_better'] >= thr and r['loss_choose_B'] < eps: return "ship B"
    if r['p_b_better'] <= 1-thr and r['loss_choose_A'] < eps: return "keep A"
    return "keep collecting"
res = ab(120, 2400, 150, 2400)
print(res); print(decide(res))

## 15.5 Project: Time-Series Forecasting Study (Naive, Exponential Smoothing, ARIMA-style)

In [ ]:
def seasonal_naive(h, hist, m=7): return np.array([hist[-m + (i % m)] for i in range(h)])
def ses(h, hist, alpha=None):
    def run(a):
        l = hist[0]; sse = 0
        for v in hist[1:]:
            sse += (v-l)**2; l = a*v + (1-a)*l
        return sse, l
    if alpha is None:
        alpha = min(np.linspace(0.05, 0.95, 19), key=lambda a: run(a)[0])
    return np.repeat(run(alpha)[1], h)
def ar_seasonal_diff(h, hist, p=2, m=7):
    d = hist[m:] - hist[:-m]
    X = np.column_stack([np.ones(len(d)-p)] + [d[p-k-1:len(d)-k-1] for k in range(p)])
    coef, *_ = np.linalg.lstsq(X, d[p:], rcond=None)
    ext = list(d); full = list(hist)
    for i in range(h):
        nd = coef[0] + sum(coef[1+k]*ext[-1-k] for k in range(p))
        ext.append(nd); full.append(full[-m] + nd)
    return np.array(full[len(hist):])

for origin in range(150, len(y)-7+1, 7):          # rolling origin
    hist, actual = y[:origin], y[origin:origin+7]
    for k, f in models.items():
        errs[k].append(np.abs(actual - f(7, hist)))
mase_scale = np.mean(np.abs(y[7:] - y[:-7]))

## 15.6 Project: Hypothesis-Testing Toolkit

In [ ]:
import numpy as np
from scipy import stats

def one_sample_t(x, mu0):
    x = np.asarray(x, float); n = len(x)
    t = (x.mean() - mu0) / (x.std(ddof=1) / np.sqrt(n))
    return t, 2 * stats.t.sf(abs(t), n - 1)

def welch_t(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    va, vb = a.var(ddof=1)/len(a), b.var(ddof=1)/len(b)
    t = (a.mean() - b.mean()) / np.sqrt(va + vb)
    df = (va + vb)**2 / (va**2/(len(a)-1) + vb**2/(len(b)-1))
    return t, df, 2 * stats.t.sf(abs(t), df)

def chi_square_independence(table):
    obs = np.asarray(table, float)
    exp = obs.sum(1, keepdims=True) * obs.sum(0, keepdims=True) / obs.sum()
    chi2 = ((obs - exp)**2 / exp).sum()
    df = (obs.shape[0]-1) * (obs.shape[1]-1)
    return chi2, df, stats.chi2.sf(chi2, df)

def one_way_anova(*groups):
    groups = [np.asarray(g, float) for g in groups]
    allx = np.concatenate(groups); k, n = len(groups), len(allx)
    ssb = sum(len(g) * (g.mean() - allx.mean())**2 for g in groups)
    ssw = sum(((g - g.mean())**2).sum() for g in groups)
    F = (ssb/(k-1)) / (ssw/(n-k))
    return F, (k-1, n-k), stats.f.sf(F, k-1, n-k)

def permutation_test(a, b, n_perm=10_000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, float), np.asarray(b, float)
    obs = a.mean() - b.mean(); pool = np.concatenate([a, b]); na = len(a)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(pool)
        count += abs(pool[:na].mean() - pool[na:].mean()) >= abs(obs) - 1e-12
    return obs, (count + 1) / (n_perm + 1)

def bonferroni(p):
    p = np.asarray(p, float); return np.minimum(p * len(p), 1.0)

def holm(p):                                   # step-down: smallest p gets the biggest multiplier
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(run, 1.0)
    return adj

def benjamini_hochberg(p):                     # step-up from the largest p
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)[::-1]
    adj = np.empty(m); run = 1.0
    for rank, i in enumerate(order):
        run = min(run, p[i] * m / (m - rank)); adj[i] = run
    return adj

In [ ]:
def test_chi_square_known_answer():
    chi2, df, p = hypo.chi_square_independence([[10, 20], [20, 10]])   # expected 15 each
    close(chi2, 4 * 25 / 15); assert df == 1

def test_anova_known_answer():
    F, df, p = hypo.one_way_anova([1, 2, 3], [2, 3, 4], [5, 6, 7])
    close(F, (26/2) / (6/6)); assert df == (2, 6)

def test_corrections_known_answer():
    p = [0.01, 0.04, 0.03, 0.005]
    assert np.allclose(hypo.bonferroni(p), [0.04, 0.16, 0.12, 0.02])
    assert np.allclose(hypo.holm(p), [0.03, 0.06, 0.06, 0.02])
    assert np.allclose(hypo.benjamini_hochberg(p), [0.02, 0.04, 0.04, 0.02])

def test_fdr_controls_false_discoveries():
    rng = np.random.default_rng(4); fdp = []
    for _ in range(200):
        p = np.concatenate([rng.uniform(size=90), rng.uniform(0, 0.001, 10)])  # first 90 are true nulls
        rej = hypo.benjamini_hochberg(p) < 0.05
        fdp.append(rej[:90].sum() / max(rej.sum(), 1))
    assert np.mean(fdp) < 0.05

## Chapter recap

## Practice set

**Task 1** (easy). Apply Bonferroni to p-values 0.01, 0.04, 0.03, 0.005. What are the adjusted values and which are below 0.05?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch15.html